<img src="images/purple_net.jpeg" alt="purple_net"> 

# **<font color="purple"> Perceptron -- Lab 2</font>**
### **<font color="purple">Prof. Tiziana Ligorio</font>**
### **<font color="purple">Hunter College of The City University of New York</font>**
---
---

# Make a copy of this notebook in your drive so you can edit it

In Lecture 3 we discussed the perceptron:

<img src="images/perceptron.png" alt="perceptron" width="400">   


# Training a Perceptron

To illustrate Perceptrons, let's look again at the [Iris Dataset](https://archive.ics.uci.edu/ml/datasets/iris) that we analyzed in [Lab1](https://colab.research.google.com/drive/15q7gkjv8F-PDlMKcQKyB8bqYl4XFkJcI?usp=sharing). It contains 3 classes of 50 instances each, where each class refers to a type of iris plant. From Lab 1 recall the attributes are: `sepal_length,  sepal_width, petal_length, petal_width, species`

<img src="images/iris.png" alt="iris" width="500"> 


ML packages such as `sklearn` and (as we will see later) `PyTorch` come prepackaged with a few small standard datasets, so we can simply load the data via the package. This will illustrate another type of dataset. You will likely work with different types of dataset objects on different projects. It is good to get used to exploring new data types.

In [1]:
import numpy as np
import torch
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [2]:
iris = load_iris() #load the data
type(iris.data) #find out what data structure we are working with

numpy.ndarray

In [3]:
# inspect the number of rows and columns of iris.data
iris.data.shape

(150, 4)

`sklearn` already separates the features from the label, thus we see 4 columns here. The targets are stored in `iris.target`

In [4]:
# inspect the number of rows and columns of iris.target
iris.target.shape

(150,)

You can also inspect the names of the features and the labels, stored in `iris.feature_names` and `iris.target_names`

In [5]:
iris.feature_names

['sepal length (cm)',
 'sepal width (cm)',
 'petal length (cm)',
 'petal width (cm)']

In [6]:
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

For both training and testing, we store our data and labels in $\bf{X}$  and $\bf{y}$, respectively. A perceptron is a binary classifier that feeds the weighted sum of its input through a step activation function and outputs either 1 or 0. We will train our perceptron to tell us whether or not an instance is Iris Setosa. Thus, we need to encode the labes as being 1 if the target is `setosa` and 0 otherwise.

In [7]:
# use binary encoding of the target for setosa or not setosa: convert iris.target to be 1s if the target is setosa, 0 otherwise
# use train_test_split() on iris.data and the binary-encoded iris.target to set aside 20% of the data for testing
# use the random_state kwa to set a random seed to 42 so every run will generate the same test set

X_train, X_test, y_train, y_test = train_test_split(iris.data,
                                                    (iris.target == 0).astype(int), test_size=0.2, random_state=42)
#where iris.target == 0 will return 1 if target is 'setosa', i.e. at position 0 in target_names, and 0 otherwise

Observe the shape of training and test data

In [8]:
X_train.shape

(120, 4)

In [9]:
X_test.shape

(30, 4)

In [10]:
y_train.shape

(120,)

In [11]:
y_test.shape

(30,)

<a name="torch_perc"></a>
## Train a Perceptron using PyTorch

Recall the training process **for each instance** $\bf{x}$:

<img src="images/perceptron.png" alt="perceptron" width="300"/>

$$                                                                                                             
  \hat{y} = \begin{cases} 1 & \text{if } \mathbf{w} \cdot \mathbf{x} = \sum_{i=0}^n w_i x_i \geq 0 \\ 0 & 
  \text{otherwise} \end{cases}                                                                                   
$$ 

 $$                                                                                                             
  \text{if } \hat{y} \neq y: \quad \forall\, i = 0, \ldots, n \quad w_i \leftarrow w_i + \eta(y - \hat{y})x_i
$$  


We implement the Perceptron from scratch using PyTorch tensors. Notice how `torch.dot` computes the weighted sum $\bf{w} \cdot \bf{x}$ efficiently using vectorized operations.

In [12]:
import torch

class TorchPerceptron:
    """Perceptron learning rule implemented with PyTorch tensors."""

    def __init__(self, n_features, learning_rate=1.0, max_iter=1000, random_state=42):
        torch.manual_seed(random_state)
        self.lr = learning_rate
        self.max_iter = max_iter
        # Initialize weights and bias to zero 
        self.weights = torch.zeros(n_features, dtype=torch.float32)
        self.bias = torch.tensor(0.0)

    def _step_activation(self, z):
        """Step activation function: output 1 if z >= 0, else 0"""
        return 1.0 if z >= 0 else 0.0

    def fit(self, X, y):
        """Train using the perceptron learning rule."""
        # Convert numpy arrays to PyTorch tensors
        X_t = torch.tensor(X, dtype=torch.float32)
        y_t = torch.tensor(y, dtype=torch.float32)

        for epoch in range(self.max_iter):
            errors = 0
            for xi, yi in zip(X_t, y_t):
                # Compute weighted sum using torch.dot (vectorized!)
                z = torch.dot(self.weights, xi) + self.bias
                y_hat = self._step_activation(z.item())
                error = yi.item() - y_hat
                if error != 0:
                    # Perceptron weight update rule: w_i <- w_i + eta * (y - y_hat) * x_i
                    self.weights += self.lr * error * xi
                    self.bias += self.lr * error
                    errors += 1
            if errors == 0:  # convergence: no misclassifications this epoch
                break
        return self

    def predict(self, X):
        """Predict class labels for samples in X. Returns a numpy array so it
        works directly with sklearn's accuracy_score and other evaluation tools."""
        X_t = torch.tensor(X, dtype=torch.float32)
        preds = []
        for xi in X_t:
            z = torch.dot(self.weights, xi) + self.bias
            preds.append(int(self._step_activation(z.item())))
        return np.array(preds, dtype=int)


In [13]:
# max_iter=1000, learning_rate=1.0, random_state=42
perc = TorchPerceptron(n_features=4, learning_rate=1.0, max_iter=1000, random_state=42)
perc.fit(X_train, y_train);

We now have a trained perceptron. Let's see how it classifies our first 10 test examples:

In [14]:
# observe the first 10 rows of your test set
X_test[:10]

array([[6.1, 2.8, 4.7, 1.2],
       [5.7, 3.8, 1.7, 0.3],
       [7.7, 2.6, 6.9, 2.3],
       [6. , 2.9, 4.5, 1.5],
       [6.8, 2.8, 4.8, 1.4],
       [5.4, 3.4, 1.5, 0.4],
       [5.6, 2.9, 3.6, 1.3],
       [6.9, 3.1, 5.1, 2.3],
       [6.2, 2.2, 4.5, 1.5],
       [5.8, 2.7, 3.9, 1.2]])

In [15]:
#predict the labels of the first 10 instances in our test set
y_hat_top10 = perc.predict(X_test[:10])

Observe and compare the predictions with the true labels:

In [16]:
# display top10 predictions
y_hat_top10

array([0, 1, 0, 0, 0, 1, 0, 0, 0, 0])

In [17]:
# display top10 true labels
y_test[:10]

array([0, 1, 0, 0, 0, 1, 0, 0, 0, 0])

It looks like our perceptron predicts with 100% accuracy. Let's evaluate using accuracy (% correct) on the full test set, where accuracy is measured as:  

$\frac{correct\_predictions}{ total\_predictions}$

In [18]:
from sklearn.metrics import accuracy_score

# predict on the full test set
y_hat = perc.predict(X_test)

# measure accuracy_score comparing true labels to predictions
accuracy = accuracy_score(y_test, y_hat)

accuracy

1.0

Indeed, the accuracy on the test set is 100%. This is a toy dataset and it is linearly separable.

Training a perceptron with a linearly separable dataset is a fairly simple task, but make sure you do understand what is going on, what a perceptron is and how it is trained.  
Now, let's review some linear algebra with PyTorch.

<a name="Notation"></a>
# Notation

Throughout these notebooks we'll use the following notation, which is standard across most deep learning literature.


* lowercase letters denote scalars (numbers)
* bold lowercase letters denote vectors
* bold uppercase letters denote matrices

Keep in mind how we are storing our data (our training examples, features and labels):  

---


$ \bf{X}=
	\begin{bmatrix}
	x_1^{(1)} & x_2^{(1)} & x_3^{(1)} & ... & x_n^{(1)} \\
	x_1^{(2)} & x_2^{(2)} & x_3^{(2)} & ... & x_n^{(2)}\\
	x_1^{(3)} & x_2^{(3)} & x_3^{(3)} & ... & x_n^{(3)}\\
    ... \\
    x_1^{(m)} & x_2^{(m)} & x_3^{(m)} & ... & x_n^{(m)}\\
	\end{bmatrix}
	\quad
	$ $\bf{y}=\begin{bmatrix}
	y^{(1)}  \\
	y^{(2)} \\
	y^{(3)} \\
    ... \\
    y^{(m)}\\
	\end{bmatrix}
	\quad$

      
* $n$ : the number of features
* $m$ : the number of instances
*  $\bf{X}$ : the training examples, an $m \times n$ matrix
* $\bf{x}_i$: the $i^{th}$ feature, a size $m$ vector
* $\bf{x}^{(i)}$: the $i^{th}$ instance, a size $n$ vector
* $x_j^{(i)}$: the $j^{th}$ feature of the $i^{th}$ instance
* $\bf{y}$ : the labels (for classification) or target value (for regression), a size $m$ vector  
* ${y}^{(i)}$: the label of the $i^{th}$ instance
* $\bf{\hat{y}}$ : the model predictions, a size $m$ vector
* ${\hat{y}}^{(i)}$: the model's prediction for the $i^{th}$ instance


---

The training process often boils down to adjusting some parameters until we get a model that fits the data satisfactorily (as measured by some cost function). Effectively, the training process looks for the parameters that minimize the cost function.   

For example, in this tutorial we looked at a simple linear regression model, where $x_j$ is the $j^{th}$ feature of the training example and the model is a weighted sum of the features:

$\hat{y} = w_0 + w_1x_1 + ... + w_nx_n$


where the parameters $\theta$ will also be stored  in a vector.  

Similarly, we looked at a perceptron, where:

$z = w_0 + w_1x_1 + w_2x_2 +  ... + w_nx_n = \bf{w}\bf{\cdot x}$

$ \hat{y}=f(z)$

where $\bf{x}$ is a row vector that holds all the feature values and $\bf{w}$ is a column vector that holds all the feature weights. These are multiplied together using matrix multiplication. You should also appreciate how the notation becomes much more compact.

So essentially, **because the data, the labels and the parameters will all be represented as vectors (1D) or matrices (2D) or tensors (nD - discussed later on), traning an ML model usually relies heavily on linear algebra.**

<img src="images/xkcd_ml_small.png" alt="xkcd" width="300">

*Image credit: xkcd*
